# 3.2.4 不要凭文件名判断模型

文件名叫 `flower-detection.onnx`，不能说明它会识别花卉。

先看 SHA-256 是否等于已核对的 ImageNet ResNet：`af16a04a6ec48ac494065d4439fe9dea590d337b9ca6dc328160ccf04a217b9c`。再看 `labels.txt` 是否有 1000 类，以及里面有没有 tench、toilet tissue 这类非花卉名称。

权重可用时可以做一次推理。softmax 得到的是置信度，不是准确率。即便最高类别碰巧是 daisy，它也只是 ImageNet 1000 类中的一个词。本题结论是：不能按文件名做花卉识别。

In [ ]:
import hashlib, json
from pathlib import Path
import numpy as np
import scipy.special
from PIL import Image
import onnxruntime as ort

KNOWN = "af16a04a6ec48ac494065d4439fe9dea590d337b9ca6dc328160ccf04a217b9c"
raw = Path("flower-detection.onnx").read_bytes()
if raw.startswith(b"version https://git-lfs.github.com/spec/v1"):
    text = raw.decode("utf-8", errors="replace")
    digest = next(line.split("sha256:", 1)[1].strip() for line in text.splitlines() if line.startswith("oid sha256:"))
else:
    digest = __________(raw).hexdigest()
print("SHA-256:", digest)
print("与 ImageNet ResNet 相同：", digest == KNOWN)

labels = [line.strip() for line in Path("labels.txt").read_text(encoding="utf-8").splitlines() if line.strip()]
print("标签数量：", len(labels))
assert len(labels) == __________, "标签不是 ImageNet 1000 类"
assert digest == KNOWN

def preprocess_image(image, resize_size=256, crop_size=224, mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)):
    image = image.resize((resize_size, resize_size), Image.BILINEAR)
    width, height = image.size
    left, top = (width - crop_size) / 2, (height - crop_size) / 2
    image = image.crop((left, top, left + crop_size, top + crop_size))
    array = np.asarray(image).astype(np.float32) / 255.0
    array = (array - np.array(mean, dtype=np.float32)) / np.array(std, dtype=np.float32)
    return np.transpose(array, (2, 0, 1)).reshape(1, 3, crop_size, crop_size)

# 只有检出真实权重时才推理。指针文件到上面的摘要核对就可以停止。
if not raw.startswith(b"version https://git-lfs.github.com/spec/v1"):
    session = ort.InferenceSession("flower-detection.onnx")
    image = Image.open("flower_test.png").convert("RGB")
    processed = preprocess_image(image).astype(np.float32)
    output = session.run(None, {session.get_inputs()[0].name: processed})[0]
    # 这里是置信度，不要把变量命名成准确率。
    confidence = __________(output, axis=-1)[0]
    predicted = labels[int(np.argmax(confidence))]
    print(predicted, float(confidence.max()))
    print("该词来自 ImageNet 标签表，不能证明这是花卉专用模型。")
